# Mass spectrometry: Pfeiffer Quadstar `.asc`

`MSData` keeps every datablock of the export as its own variable: `block_0` holds
the ion currents over `(cycle, mz)`, further blocks hold auxiliary sensors (here
the PKR pressure gauge). The example is the test fixture under `tests/massspec/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
ASC = next((Path("..") / "tests" / "massspec").glob("*.asc"))
tos_start = pd.Timestamp("2026-03-26 14:57:59", tz="Europe/Amsterdam")   # start time in the file header
ms_raw = ldp.MSData.from_quadstar_asc(ASC, tos_start=tos_start)
ms_raw

`tos_start` anchors the elapsed time `tos` to the wall clock. Without it,
`tos` still counts from the first cycle, but `timestamps` is `None`. The times
of the file are read in `Europe/Amsterdam` by default (`tz_str`).

In [ ]:
print("m/z channels:", ms_raw.mz)
print("blocks:", ms_raw.block_ids)
print("cycles:", ms_raw.n_cycle)
print("tos_start:", ms_raw.tos_start)

## Processing

`correct_traces` shifts negative traces up to zero and `mask_overloaded` turns
detector saturation values into NaN. Both return a new object and are logged.

In [ ]:
ms = ms_raw.correct_traces().mask_overloaded()
print(ms.format_history())

## Traces

`get_trace` returns one m/z channel along `cycle`, with `tos` as a coordinate;
`rolling_window` smooths before `normalize` scales the trace to its maximum.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for mz in (2, 18, 28, 44):
    trace = ms.get_trace(mz, rolling_window=5)
    ax.semilogy(trace["tos"] / 3600, trace, label=f"m/z {mz}")
ax.set_xlabel("time on stream / h")
ax.set_ylabel("ion current / A")
ax.legend()
plt.show()

## An auxiliary block

The pressure gauge is channel 0 of block 1.

In [ ]:
pressure = ms.get_channel(1, 0)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(pressure["tos"] / 3600, pressure)
ax.set_xlabel("time on stream / h")
ax.set_ylabel(f"pressure / {ms.unit(1)}")
plt.show()

## A time window

`select_tos_range` cuts in elapsed seconds; `select_timestamp_range` does the
same in wall-clock time (a naive string is read in the timezone of the data).

In [ ]:
start = ms.timestamps[0] + pd.Timedelta(hours=1)
window = ms.select_timestamp_range(start, start + pd.Timedelta(hours=2))
print(window.n_cycle, "cycles from", window.timestamps[0], "to", window.timestamps[-1])